In [1]:
%matplotlib inline

import os
import sys
import copy

import matplotlib.pyplot as plt

sys.path.append('../../../')

%load_ext autoreload
%autoreload 2

from computer_vision.yolov11_pose.parameter_parser import parser
from computer_vision.yolov11_pose.nn.tasks import PoseModel
from computer_vision.yolov11_pose.utils.metrics import DetMetrics

In [2]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Any

import numpy as np
import torch

from computer_vision.yolov11_pose.cfg import get_cfg
from computer_vision.yolov11_pose.utils.checks import check_imgsz

class DectectionValidator:
    """
    This class implements validation functionality specific to objet detection tasks, including metrics calculation, prediection,
    processing, and visualization of results

    Examples:
        >>> args=dict(model='yolo11n.pt', data='coco8.yaml')
        >>> validator=DetectionValidator(args=args)
        >>> validator()
    """
    def __init__(self, dataloader=None, save_dir=None, args=None)->None:
        """Initialize detection validator with necessary variables and settings
        Args:
            dataloader (torch.utils.data.DataLoader, optional): Dataloader to use for validation
            save_dir (Path, optional): Directory to save results
            args (dict[str, Any], optional): Arguments for validator
        """
        self.args=get_cfg(overrides=args)
        self.dataloader=dataloader
        self.stride=None
        self.data=None
        self.device=None
        self.batch_i=None # current batch index
        self.training=True # whether the model is in training mode
        self.names=None # class name mapping
        self.seen=None # number of images seen so far during validation
        self.stats=None # statistics collected during validation
        self.confusion_matrix=None
        self.jdict=None # list to store JSON validation results
        self.speed={'preprocess':0., 'inference':0., 'postprocess':0} # storing respective batch processing time in milliseconds
        self.save_dir=self.args.save_dir
        self.save_dir.mkdir(parents=True, exist_ok=True)
        if self.args.conf is None: self.args.conf=0.01 if self.args.task=='obb' else 0.001 # reduce OBB val memory usage
        self.args.imgsz=check_imgsz(self.args.imgsz, max_dim=1)

        self.plots={}
        self.is_coco=False
        self.is_lvis=False
        self.class_map=None
        self.args.task='detect'
        self.iouv=torch.linspace(0.5, 0.95, 10) # IoU thresholds from .5 to .95 in spaces of .05, i.e.,  mAP@0.5:0.95
        self.niou=self.iouv.numel()
        self.metrics=DetMetrics()
        


overrides  <class 'dict'>  cfg  <class 'dict'>


'D:/results/yolov11_pose/validation'

In [ ]:
class PoseValidator(DectectionValidator):
    """A class extending the DetectionValidator class for validation based on a pose model

    This validator is specifically designed for pose estimation tasks, handling keypoints and implementing specialized metrics
    for pose evaluation
    
    Examples:
        >>> args=dict(model='yolo11n-pose.pt', data='coco8-pose.yaml')
        >>> validator=PoseValidator(args=args)
        >>> validator()
    """
    def __init__(self, dataloader=None, save_dir=None, args=None)->None:
        """Initialize a PoseValidator object for pose estimation validation
        
        The validator is specifically designed for pose estimation tasks, handling keypoints and implementing specialized metrics
        for pose evaluation

        Args:
            dataloader(torch.utils.data.DataLoader, optional): Dataloader to be used for validation
            save_dir (Path|str, optional): Directory to save results
            args (dict|Namespace, optional): Arguments for the validator including task set to `pose`
        Examples:
            >>> args=dict(model='yolo11n-pose.pt', data='coco8-pose.yaml')
            >>> validator=PoseValidator(args=args)
            >>> validator()
        Notes:
            This class extends DetectionValidator with pose-specific functionality. It initializes with sigma values
            for OKS calculation and sets up PoseMetrics for evaluation. A warning is displayed when using Apple MPS due 
            to a known bug with pose models 
        """
        super().__init__(dataloader, save_dir, args)
        self.sigma=None
        self.kpt_shape=None
        self.args.task='pose'
        self.metrics=PoseMetrics()
        if isinstance(self.args.device, str) and self.args.device.lower()=='mps':
            warnings.warn("Apple MPS known Pose bug. Recommend `device=cpu` for Pose Model"
                          "See https://github.com/ultralytics/ultralytics/issues/4031.")

In [ ]:
output_dirpath='D:/results/yolov11_pose/validation'
args=parser.parse_args(f'--save-dir {output_dirpath}'.split())
validator=DectectionValidator(dataloader=None, args=args)
validator.args.save_dir